In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mrl_trace import paths
from mrl_trace.stats import bootstrap_ci

# QUICK=True re-runs a fast, few-seed version of an experiment in-kernel (serial, no Pool);
# the default (QUICK=False) REPLAYS the committed 20-seed grid so every figure renders
# instantly and deterministically. The heavy sweeps document a `--full` shell command instead.
QUICK = False

GREEN, INDIGO, RED, GOLD, GREY = "#3aa07a", "#2f4b8f", "#c0392b", "#e0a93b", "#9aa6b2"
# sequential viridis by retention (long -> short), matching the manuscript window figures
VIR = [plt.cm.viridis(x) for x in (0.85, 0.5, 0.15)]

def _clean(ax):
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    ax.set_axisbelow(True); ax.grid(True, color="0.88", lw=0.5)

print("data/results:", paths.results_dir())

In [ ]:
if QUICK:
    from mrl_trace.bandit import run_learning_and_window
    r = run_learning_and_window(seeds=6)
else:
    r = paths.load_result("tier3_results.npy")

delays = np.asarray(r["delays"], float)
D0 = r["D0"]
W = 50                                   # running-mean window (matches gen_rl_curve.py)

def running(rw_2d, w=W):
    cs = np.cumsum(np.insert(rw_2d, 0, 0.0, axis=1), axis=1)
    rr = (cs[:, w:] - cs[:, :-w]) / w    # (seeds, trials-w)
    return rr.mean(0), np.percentile(rr, 2.5, axis=0), np.percentile(rr, 97.5, axis=0)

fig, (axA, axB) = plt.subplots(1, 2, figsize=(9.6, 3.6))

# (a) learning curve: device vs no-trace, running mean + 95% CI band
for arr, c, lab in [(r["curve_device"], GREEN, "device trace"),
                    (r["curve_notrace"], GREY, "no-trace")]:
    m, lo, hi = running(np.asarray(arr, float))
    x = np.arange(W, W + len(m))
    axA.plot(x, m, color=c, lw=1.7, label=lab, zorder=4)
    axA.fill_between(x, lo, hi, color=c, alpha=0.2, lw=0, zorder=2)
axA.axhline(0.5, ls="--", color=RED, lw=1.0, zorder=1, label="chance")
axA.set_xlabel("trial"); axA.set_ylabel("reward rate")
axA.set_ylim(0.3, 1.03); axA.set_title(rf"(a) learning curve ($D_0={D0:g}$ s)", fontsize=10)
axA.legend(fontsize=8, frameon=False, loc="lower right"); _clean(axA)

# (b) delay x retention window
tau_style = [(10.0, VIR[0]), (2.0, VIR[1]), (0.5, VIR[2])]
for tl, c in tau_style:
    y = np.asarray(r["reward_rate"][tl], float)
    ci = r["reward_rate_ci"][tl]
    lo = np.array([b[0] for b in ci]); hi = np.array([b[1] for b in ci])
    axB.plot(delays, y, marker="o", ms=4, lw=1.6, color=c,
             label=rf"$\tau_{{leak}}={tl:g}$ s", zorder=4)
    axB.fill_between(delays, lo, hi, color=c, alpha=0.18, lw=0, zorder=2)
axB.axhline(0.75, ls=":", color=GREY, lw=1.0); axB.axhline(0.5, ls="--", color=RED, lw=0.9)
axB.set_xscale("log"); axB.set_xticks(delays)
axB.set_xticklabels([f"{d:g}" for d in delays], fontsize=8)
axB.set_xlabel(r"action$\to$reward delay $D$ (s)"); axB.set_ylabel("final reward rate")
axB.set_ylim(0.4, 1.03); axB.set_title("(b) delay x retention window", fontsize=10)
axB.legend(fontsize=8, frameon=False, loc="lower left"); _clean(axB)
plt.show()

df, nf = r["device_final"], r["notrace_final"]
print(f"device final reward rate  {df[0]:.3f}  (95% CI {df[2]:.3f}-{df[3]:.3f})")
print(f"no-trace final            {nf[0]:.3f}  (95% CI {nf[2]:.3f}-{nf[3]:.3f})")
print("max learnable delay per tau_leak (s):", {f"{k:g}": v for k, v in r["max_learn"].items()})
print(f"C1 no-trace at chance: {'PASS' if nf[0] <= 0.5 + 0.10 else 'fail'}   "
      f"C2 device >= criterion: {'PASS' if df[0] >= 0.75 else 'fail'}")

In [ ]:
if QUICK:
    from mrl_trace.bandit import run_scaling
    r = run_scaling(seeds=6)
else:
    r = paths.load_result("tier4_results.npy")

cells_sa = [(2, 2), (4, 2), (4, 4), (8, 4), (8, 8), (12, 8)]
labels = [f"{S}x{A}" for S, A in cells_sa]

def _ci_err(entries):
    m = np.array([e[0] for e in entries])
    lo = np.array([e[2] for e in entries]); hi = np.array([e[3] for e in entries])
    return m, np.vstack([m - lo, hi - m])

dev, dev_err = _ci_err([r[c]["device"] for c in cells_sa])
nt, nt_err = _ci_err([r[c]["no_trace"] for c in cells_sa])
chance = np.array([r[c]["chance"] for c in cells_sa])
crit = np.array([r[c]["crit"] for c in cells_sa])
passed = np.array([r[c]["passed"] for c in cells_sa])
x = np.arange(len(cells_sa))

fig, ax = plt.subplots(figsize=(6.6, 3.8))
ax.plot(x, chance, "--", color=RED, lw=1.1, label=r"chance ($1/A$)", zorder=2)
ax.plot(x, crit, ":", color="0.5", lw=1.2, label="criterion", zorder=2)
ax.errorbar(x, dev, yerr=dev_err, marker="o", ms=5, lw=1.6, color=GREEN,
            capsize=2.5, label="device trace", zorder=5)
ax.errorbar(x, nt, yerr=nt_err, marker="s", ms=4, lw=1.3, color=GREY,
            capsize=2.5, label="no-trace", zorder=4)
for xi, dd, ok in zip(x, dev, passed):
    if ok:
        ax.scatter([xi], [dd], s=110, facecolors="none", edgecolors=GREEN, lw=1.7, zorder=6)
firstfail = int(np.argmax(~passed)) if np.any(~passed) else len(cells_sa)
if firstfail < len(cells_sa):
    ax.axvspan(firstfail - 0.5, len(cells_sa) - 0.5, color=RED, alpha=0.06, zorder=0)
    ax.text((firstfail + len(cells_sa) - 1) / 2, 0.93, "not converged\nin budget",
            fontsize=7.5, ha="center", va="top", color=RED, style="italic")
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_xlabel(r"array size $S\times A$"); ax.set_ylabel("final reward rate")
ax.set_ylim(0, 1.02)
ax.legend(fontsize=8, frameon=False, loc="lower left", handlelength=1.6)
ax.set_title("Scaling: device converges up to ~4 actions, then a sample-efficiency wall")
_clean(ax); plt.show()

for c, lab in zip(cells_sa, labels):
    e = r[c]
    ttc = e["trials_to_crit"]
    print(f"  {lab:6s}: device {e['device'][0]:.3f}  crit {e['crit']:.3f}  "
          f"converged {e['n_converged']}/{e['n_seeds']}  "
          f"trials_to_crit {ttc if ttc is not None else 'n/a'}  "
          f"{'PASS' if e['passed'] else 'fail'}")

In [ ]:
if QUICK:
    from mrl_trace.bandit import run_remedies
    r = run_remedies(seeds=6)
else:
    r = paths.load_result("tier5_results.npy")

keys = list(r.keys())                     # preserves R0..R4 insertion order
short = ["R0\nbaseline", "R1\n4x budget", "R2\ndirected\nexpl.",
         "R3\nabstract\ntrace", "R4\nbudget+\ndirected"]
vals = np.array([r[k]["final"][0] for k in keys])
lo = np.array([r[k]["final"][2] for k in keys]); hi = np.array([r[k]["final"][3] for k in keys])
err = np.vstack([vals - lo, hi - vals])
psd = np.array([r[k]["passed"] for k in keys])
isabs = np.array(["abstract" in k for k in keys])   # abstract-trace row grey, device rows green
bar_c = [GREY if a else GREEN for a in isabs]
xb = np.arange(len(keys))
crit88 = 0.5 * (1 + 1.0 / 8)

fig, ax = plt.subplots(figsize=(6.6, 3.8))
ax.bar(xb, vals, yerr=err, color=bar_c, capsize=3, width=0.66, edgecolor="white", zorder=3)
ax.axhline(crit88, ls=":", color="0.5", lw=1.2, zorder=2, label="criterion")
ax.axhline(1.0 / 8, ls="--", color=RED, lw=1.1, zorder=2, label="chance")
for xi, v, h, ok in zip(xb, vals, hi, psd):
    ax.text(xi, h + 0.03, "PASS" if ok else "fail", fontsize=7.5, ha="center",
            color=(GREEN if ok else RED), fontweight=("bold" if ok else "normal"))
ax.set_xticks(xb); ax.set_xticklabels(short, fontsize=8)
ax.set_ylabel("final reward rate"); ax.set_ylim(0, 1.12)
ax.legend(fontsize=8, frameon=False, loc="upper left", handlelength=1.6)
ax.set_title("Remedies on the failing 8x8 case"); _clean(ax); plt.show()

for k in keys:
    e = r[k]; print(f"  {'PASS' if e['passed'] else 'fail'}  {e['final'][0]:.3f}  {k}")

In [ ]:
if QUICK:
    from mrl_trace.maze import run_sequential
    r = run_sequential(seeds=6)
else:
    r = paths.load_result("exp6_sequential.npy")

crit, chance = r["crit"], r["chance"]
taus = np.asarray(r["taus"], float)
grid_delays = np.asarray(r["delays"], float)
grid = np.asarray(r["grid"], float)
grid_ci = np.asarray(r["grid_ci"], float)
dmax = r["dmax"]

fig, (axA, axB) = plt.subplots(1, 2, figsize=(10, 3.8))

# (a) learning curves (curves are already the seed-mean running rate)
cond_style = [("device", GREEN, "device trace"), ("rstdp", INDIGO, r"R-STDP (matched $\tau$)"),
              ("eprop", GOLD, "reward-based e-prop"), ("no_trace", GREY, "no-trace")]
for name, c, lab in cond_style:
    if name not in r["curves"]:
        continue
    cur = np.asarray(r["curves"][name], float)
    axA.plot(np.arange(len(cur)), cur, color=c, lw=1.6, label=lab, zorder=4)
axA.axhline(crit, ls=":", color=GREY, lw=1.0); axA.axhline(chance, ls="--", color=RED, lw=0.9)
axA.set_xlabel("episode (running mean)"); axA.set_ylabel("reward rate")
axA.set_ylim(0.3, 1.03)
axA.set_title(rf"(a) learning curves ($D_0={r['D0']:g}$ s, $\tau={r['TAU0']:g}$ s)", fontsize=10)
axA.legend(fontsize=8, frameon=False, loc="lower right"); _clean(axA)

# per-condition finals + CI (committed grid stores per-seed finals only)
for name, _, _ in cond_style:
    if name in r["finals"]:
        f = np.asarray(r["finals"][name], float); lo, hi = bootstrap_ci(f)
        print(f"  {name:9s} final {f.mean():.3f}  (95% CI {lo:.3f}-{hi:.3f})")
pol = np.asarray(r["policy"], float)
print(f"  device policy-correctness {pol.mean():.3f}  (C3 {'PASS' if pol.mean() >= crit else 'fail'})")

# (b) retention x delay grid + D_max crossings
for i, (tau, c) in enumerate(zip(taus, VIR)):
    y = grid[i]; lo = grid_ci[i, :, 0]; hi = grid_ci[i, :, 1]
    axB.plot(grid_delays, y, marker="o", ms=4, lw=1.6, color=c,
             label=rf"$\tau_{{leak}}={tau:g}$ s ($D_{{max}}={dmax[tau]:g}$ s)", zorder=4)
    axB.fill_between(grid_delays, lo, hi, color=c, alpha=0.18, lw=0, zorder=2)
    dm = dmax[tau]
    if dm:
        axB.scatter([dm], [grid[i, list(r["delays"]).index(dm)]], s=90, facecolors="none",
                    edgecolors=c, lw=1.8, zorder=6)
axB.axhline(crit, ls=":", color=GREY, lw=1.0); axB.axhline(chance, ls="--", color=RED, lw=0.9)
axB.set_xscale("log"); axB.set_xticks(grid_delays)
axB.set_xticklabels([f"{d:g}" for d in grid_delays], fontsize=8)
axB.set_xlabel(r"action$\to$reward delay $D$ (s)"); axB.set_ylabel("final reward rate")
axB.set_ylim(0.4, 1.03)
axB.set_title(r"(b) retention law: $D_{max}$ grows with $\tau_{leak}$", fontsize=10)
axB.legend(fontsize=7.5, frameon=False, loc="lower left"); _clean(axB)
plt.show()

nt_mean = np.asarray(r["finals"]["no_trace"], float).mean()
k1 = bool((grid > nt_mean + 0.1).any())
print(f"D_max(tau_leak): {{ {', '.join(f'{k:g}:{v:g}' for k, v in dmax.items())} }} s")
print(f"C1 no-trace at chance: {'PASS' if nt_mean <= chance + 0.10 else 'fail'}   "
      f"C4 D_max grows with tau: {'PASS' if (dmax[20.0] >= dmax[5.0] >= dmax[2.0] and dmax[20.0] > dmax[2.0]) else 'fail'}   "
      f"K1 device exceeds no-trace: {'PASS' if k1 else 'fail'}")

In [ ]:
# Uncomment to launch the full sweeps as subprocesses (streamed; heavy -- minutes each):
# import subprocess, sys
# subprocess.run([sys.executable, "-m", "mrl_trace.bandit", "--bandit", "--full"])
# subprocess.run([sys.executable, "-m", "mrl_trace.maze", "--exp6", "--full"])
print("see the markdown above for the full-scale commands")